# Q32: Weight Initialization Scenarios
**Topic:** Deep-learning | **Difficulty:** Medium | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
What happens when you initialise your network weights with: (a) all zeros, (b) all constants (c) all random?

## Detailed Answer

### (a) All Zeros
**Problem: Symmetry** — every neuron computes the same output, receives the same gradient.
- All neurons in a layer are identical → they learn the same thing
- Network has effectively 1 neuron per layer
- **Never do this** for weights (biases at zero is fine)

### (b) All Constants (non-zero)
**Same problem as zeros** — symmetry is not broken. Every neuron still computes the same thing.
- Slightly better than zero if constant is well-chosen (activations don't die)
- But still fundamentally broken — no diversity in learned features

### (c) Random Initialization
**Correct approach** — breaks symmetry. But scale matters!

| Method | Formula | Best For |
|--------|---------|----------|
| **Xavier/Glorot** | $W \sim \mathcal{N}(0, \frac{2}{n_{in}+n_{out}})$ | Sigmoid, Tanh |
| **He/Kaiming** | $W \sim \mathcal{N}(0, \frac{2}{n_{in}})$ | ReLU, Leaky ReLU |
| **LeCun** | $W \sim \mathcal{N}(0, \frac{1}{n_{in}})$ | SELU |

**Too small** random: gradients vanish (signals shrink layer by layer)
**Too large** random: gradients explode (signals grow layer by layer)
**Xavier/He**: Maintains variance of activations across layers — the Goldilocks zone

### Modern Practices
- **Transformers**: Usually Xavier or specific init per sublayer
- **Pre-trained models**: Use pre-trained weights — initialization is moot
- **BatchNorm**: Reduces sensitivity to initialization (but doesn't eliminate it)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_forward(init_type, n_layers=10, n_neurons=256, input_dim=256):
    x = np.random.randn(1, input_dim)
    activations = []
    for i in range(n_layers):
        if init_type == 'zeros':
            W = np.zeros((n_neurons, n_neurons if i > 0 else input_dim))
        elif init_type == 'constant':
            W = np.ones((n_neurons, n_neurons if i > 0 else input_dim)) * 0.01
        elif init_type == 'random_large':
            W = np.random.randn(n_neurons, n_neurons if i > 0 else input_dim) * 1.0
        elif init_type == 'xavier':
            fan = n_neurons if i > 0 else input_dim
            W = np.random.randn(n_neurons, fan) * np.sqrt(2.0 / (fan + n_neurons))
        elif init_type == 'he':
            fan = n_neurons if i > 0 else input_dim
            W = np.random.randn(n_neurons, fan) * np.sqrt(2.0 / fan)
        x = np.maximum(0, x @ W.T)  # ReLU
        activations.append(np.mean(np.abs(x)))
    return activations

np.random.seed(42)
fig, ax = plt.subplots(figsize=(10, 5))
for init, style in [('zeros', 'k--'), ('random_large', 'r-'), ('xavier', 'b-'), ('he', 'g-')]:
    acts = simulate_forward(init)
    ax.plot(acts, style, linewidth=2, label=init)
ax.set_xlabel('Layer')
ax.set_ylabel('Mean Activation Magnitude')
ax.set_title('Effect of Weight Initialization')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways
- **Never** initialize all weights to zeros or constants — symmetry kills learning
- Use **He init** for ReLU — it's the default in PyTorch for conv/linear layers
- Use **Xavier** for sigmoid/tanh activations
- In practice, **pre-trained models** bypass initialization entirely